# Project K, Series Three: the property patch, Qwen2.5-Omni-3B

**What this is.** The causal step the council ordered. The
probes say pitch and timing stay readable at every layer; the
behavioral work says the model's own lean ranks them almost
perfectly while its spoken answers default wrong. This battery
asks the causal question: if we paste a donor tone's insides
into a different tone's run, at one layer, does the lean move
toward the donor's property?

- **Arms.** Pitch (low recipient, high donor, and the reverse)
  and onset (early recipient, late donor, and the reverse).
  Donors are independent clips, matched on the other property.
- **Patch site.** The sound's own positions, one layer at a
  time: 0, 14, 16, 18, 19, 20, 22, 24, 28, 34. (Layer 35 is
  left out on purpose: the index check showed its state is the
  polished one, and pasting it double-polishes.)
- **Readouts.** Both, under the original questions: the yes
  share (the lean) and the first spoken word. Each arm also
  reads the other property's lean, which should NOT move if
  the patch is specific.
- **Controls.** Sham pastes (donor with the SAME property
  value: the lean should stay) and post-position pastes (the
  text positions after the sound, at layers 0, 19, 28: a strict
  subset that cannot rebuild the hybrid state).
- **Open form.** The pitch arm's low-to-high pairs also get
  the open question at layers 14, 19, 24, the partial band.

**Where it runs.** Free Colab account (kadjei1). **How.** Run
the cells from the top, in order. Everything prints live and
is also written to `proppatch_run_log.txt` in the Drive
folder `tracing-qwen3b`.

**The one rule.** We claim only what we ran.


## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Tracing addition.** At the end, the cell proves the
stack it just installed: it prints the imported
transformers version and the file it loaded from, and
stops the run if the version is not the pinned 4.52.4.
It also proves scikit-learn, the library the helper
classifiers are built with: if it is missing, the cell
installs it, and either way it prints the version it
found. This notebook's model runs under the session's
own Python, so the versions printed here are the run's
versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Tracing addition: prove the installed stack --------------------------
# The model and the probes run under this same session
# Python, so the versions installed above are the versions
# the run uses. Prove them here, and stop if the
# transformers pin is not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")

# The helper classifiers are scikit-learn logistic
# regressions. Colab sessions normally ship scikit-learn;
# install it only if it is missing, and print the version
# either way so the run record names what trained the
# helpers.
try:
    import sklearn
    print("scikit-learn version:", sklearn.__version__)
except ImportError:
    print("scikit-learn missing; installing scikit-learn==1.6.1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.6.1"], check=False)
    import sklearn
    print("scikit-learn version:", sklearn.__version__)


## Step 2 · Get the clips and the masters

The battery needs the step-5 property files (clips plus
`property_manifest.csv`) from the Drive folder
`tracing-qwen3b`, and the master sounds for the head's token
check. The next cell brings the masters; the one after brings
the property files.


In [ ]:
# --- Get the two master sounds ---------------------------------------------------
import os, shutil, subprocess

FILES = ["codec_tone_ref_raw.wav", "codec_speech_raw.wav"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/codec-control"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/codec-control"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

missing = []
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.2f} MB{extra})")
    else:
        missing.append(name)
        print("MISSING:", name)
if missing:
    print()
    print("!" * 72)
    print("STOP. These master files are missing:")
    for name in missing:
        print("   ", name)
    print("Check the Drive folder name is exactly codec-control, or")
    print("upload the files by hand, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: master files missing in Step 2")
print()
print("ready: both masters (tone, speech)")

In [ ]:
# --- Get the property clips and manifest -----------------------------------
import os, shutil

WORK = "/content/property_clips"
DRIVE_CLIPS = "/content/drive/MyDrive/tracing-qwen3b/property_clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

names = sorted(n for n in os.listdir(DRIVE_CLIPS)
               if n.endswith(".wav"))
print("property clips on Drive:", len(names))
copied = 0
for name in names:
    dest = os.path.join(WORK, name)
    if not os.path.exists(dest):
        shutil.copy(os.path.join(DRIVE_CLIPS, name), dest)
        copied += 1
print("copied:", copied, "| in place:", len(os.listdir(WORK)))

src = "/content/drive/MyDrive/tracing-qwen3b/property_manifest.csv"
shutil.copy(src, "/content/property_manifest.csv")
print("manifest copied:", os.path.getsize(
    "/content/property_manifest.csv"), "bytes")


## Step 3 · Write the runner, then check

Check mode runs the proof, the loading candidates, and the
token check, then stops.


In [ ]:
# --- Write the tracing round 2 runner script -------------------------------------
# The runner holds the proof, the loading candidates, the
# token check, the clip generation, the extraction, the
# helpers, and the attention readout. It is written to a
# plain file here and only ever run under this session's
# own Python, the interpreter Step 1 installed into.
RUNNER_SOURCE = '\n# qwen_proppatch_run.py - the tracing round 2 runner for\n# Qwen2.5-Omni-3B. Written by\n# project_k_qwen_proppatch_colab.ipynb. It runs under the\n# notebook session\'s own Python: the Qwen notebooks install\n# their stack into the session (Step 1), and there is no\n# private environment to run under. Modes: "check" runs the\n# proof, the loading candidates, and the TOKEN CHECK, then\n# stops. "battery" does all of that and then generates the\n# probe clips, extracts the per-layer features, trains the\n# helpers (two readouts, 5-shuffle controls), reads the\n# attention subset, and saves every output.\n#\n# Exit codes, one per failure family, all loud:\n#   3  the stack proof failed (transformers is not 4.52.4)\n#   4  both loading candidates failed\n#   7  a feed returned no audio (never padded, never scored)\n#   9  probe clip generation failed its own measurements\n#   10 the extraction count does not match the manifest count\n#   11 the token check failed: the audio positions could\n#      not be settled from the histogram, or their count is\n#      implausible, or the thinker returned no hidden states\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nimport sklearn\nprint("scikit-learn version:", sklearn.__version__)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\nprint("attention implementation note: the loading candidates below pass")\nprint("attn_implementation=\'eager\'. The attention readout is a required")\nprint("output of this build, and the default sdpa kernel does not return")\nprint("attention weights. The sequences here are short, so the memory")\nprint("cost is small, and the hidden-state math is the same.")\n\n# --- Homes for the outputs -------------------------------------------------------\n# The tracing outputs live in a Drive folder named\n# tracing-qwen3b, created here if it is missing, with /content\n# as the second home, the house pattern.\nDRIVE_TRACING = "/content/drive/MyDrive/tracing-qwen3b"\nLOCAL_TRACING = "/content/tracing-qwen3b"\nfor d in (DRIVE_TRACING, LOCAL_TRACING):\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\ndef out_paths(name):\n    return [os.path.join(DRIVE_TRACING, name),\n            os.path.join(LOCAL_TRACING, name)]\n\ndef save_both(name, write_fn):\n    saved = []\n    for p in out_paths(name):\n        try:\n            write_fn(p)\n            saved.append(p)\n            print("saved:", p)\n        except Exception as e:\n            print("could not save", p, "-", str(e)[:160])\n    return saved\n\n# --- The load log ------------------------------------------------------------------\nimport traceback, datetime\n\nLOAD_LOG_PATHS = out_paths("proppatch_load_log.txt")\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B tracing round 2 load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the codec runner\'s candidates ---------------------------------\n# Same candidates, same order, same talker-off call as\n# round 1. The one printed difference is\n# attn_implementation="eager" (see the proof lines above\n# for why).\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto",\n            attn_implementation="eager")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\nTHINKER = model.thinker\nprint("thinker:", type(THINKER).__name__)\n\n# --- The feed path -----------------------------------------------------------------\n# Sound alone, the codec control\'s raw arm form: the\n# conversation carries an audio element naming the WAV file\n# and a text element, and no image and no video element.\n# process_mm_info decodes the audio element with the same\n# librosa call at 16 kHz mono it uses for a video\n# soundtrack. The text is the Series Two presence question\n# for the clip\'s sound, fixed per class, byte-identical to\n# round 1: the tracing walks the model\'s insides while it\n# hears the clip under the same question the outside tests\n# asked. Silence clips carry the tone question, the primary\n# task\'s question, exactly as round 1\'s negatives did, and\n# the manifest records the exact prompt on every clip.\nimport json, math\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nCLIPS = "/content/clips"\nPROMPTS = {\n    "tone": ("Is there the sound of a tone in this clip? "\n             "Answer yes or no."),\n    "speech": ("Is there the sound of speech in this clip? "\n               "Answer yes or no."),\n    "silence": ("Is there the sound of a tone in this clip? "\n                "Answer yes or no."),\n}\n\nclass FeedError(RuntimeError):\n    pass\n\ndef conversation_for(path, prompt):\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "audio", "audio": path},\n            {"type": "text", "text": prompt},\n        ],\n    }]\n\ndef feed_inputs(path, prompt, expect_silence=False):\n    # Run the feed path for one clip and build the model\'s\n    # inputs. Raises FeedError if the path returns no audio:\n    # a missing track stops the run, it is never padded. A\n    # track that decodes to all zeros also stops the run,\n    # unless the caller declares the clip silent by design\n    # (a generated true-silence clip): an all-zeros feed on\n    # a clip that should carry sound is a broken feed, not a\n    # quiet clip, and scoring it anyway is how silent\n    # failures get into the record.\n    if not os.path.exists(path):\n        raise FeedError(f"feed file missing: {path}")\n    conversation = conversation_for(path, prompt)\n    audios, images, videos = process_mm_info(\n        conversation, use_audio_in_video=False)\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if len(audio_np) == 0:\n        raise FeedError(\n            f"decoded track is empty for {path}")\n    if not expect_silence and not np.any(audio_np != 0):\n        raise FeedError(\n            f"decoded track is all zeros for {path}, on a clip "\n            f"that should carry sound (decoded length "\n            f"{len(audio_np)}, decoded rms "\n            f"{rms_db_np(audio_np):.2f} dB)")\n    text = processor.apply_chat_template(\n        conversation, add_generation_prompt=True, tokenize=False)\n    inputs = processor(text=text, audio=audios,\n                       return_tensors="pt", padding=True)\n    inputs = inputs.to(model.device).to(model.dtype)\n    return inputs, audio_np\n\ndef thinker_forward(inputs, want_attentions=False):\n    # One frozen pass through the thinker with every layer\'s\n    # numbers saved. The model is never trained, never asked\n    # to generate; this is a reading, not a conversation.\n    with torch.no_grad():\n        out = THINKER(**inputs, output_hidden_states=True,\n                      output_attentions=want_attentions)\n    return out\n\n# --- Run log: every print also lands in a file in the Drive\n# folder, so a stopped run\'s evidence never depends on the\n# notebook output area\'s scrolling.\nclass _Tee:\n    def __init__(self, *streams):\n        self.streams = streams\n    def write(self, s):\n        for st in self.streams:\n            try:\n                st.write(s)\n            except Exception:\n                pass\n        return len(s)\n    def flush(self):\n        for st in self.streams:\n            try:\n                st.flush()\n            except Exception:\n                pass\ntry:\n    _log_fh = open("/content/drive/MyDrive/tracing-qwen3b/"\n                   "proppatch_run_log.txt", "w", buffering=1)\nexcept Exception:\n    _log_fh = open("/content/proppatch_run_log.txt", "w",\n                   buffering=1)\nsys.stdout = _Tee(sys.stdout, _log_fh)\n\n# --- The token check -----------------------------------------------------------------\n# Round 2 fix 1. Round 1\'s gate picked the first candidate\n# id that decoded to the audio pad token and occurred at\n# least once, and it occurred exactly once: the probes may\n# have pooled over a single marker position instead of the\n# audio positions. This check settles the question from the\n# data, in the open, before any extraction:\n#\n#   * for one tone clip and one speech clip (the masters),\n#     print the FULL histogram of token ids in the processed\n#     input row: every id with count > 0, with the token\'s\n#     name and its decoded form, special tokens flagged;\n#   * print the processor\'s audio feature lengths (the\n#     input_features shape and the feature_attention_mask\n#     sums), the independent record of how many audio frames\n#     the processor built;\n#   * collect the audio-token candidates exactly as round 1\n#     did (tokenizer tokens whose text mentions audio, the\n#     config attributes, the converted pad id), print each\n#     candidate\'s count in the row, and DERIVE the pooling\n#     id from the histogram: the candidate with the highest\n#     count, the one that occurs at audio-frame scale;\n#   * both clips must derive the same id, and its count\n#     must be at least MIN_AUDIO_POSITIONS, or the step\n#     prints TOKEN CHECK FAILED and stops the run (exit 11)\n#     before any extraction;\n#   * one thinker forward on the tone clip then confirms\n#     the hidden states come back and fixes the layer count\n#     and hidden size the extraction will use.\n# Every line of the check is also collected and saved to\n# proppatch_token_check.txt, pass or fail, so the histogram\n# is in the record either way.\nfrom collections import Counter\n\nMIN_AUDIO_POSITIONS = 10\nAUDIO_TOKEN_ID = None   # set by the token check, from the histogram\nN_STATES = None\nHIDDEN_SIZE = None\nTOKEN_CHECK_LINES = []\n\ndef tc(line=""):\n    print(line)\n    TOKEN_CHECK_LINES.append(line)\n\ndef save_token_check():\n    text = "\\n".join(TOKEN_CHECK_LINES) + "\\n"\n    def w(p):\n        with open(p, "w") as fh:\n            fh.write(text)\n    save_both("proppatch_token_check.txt", w)\n\ndef token_check_fail(reason):\n    tc()\n    tc("!" * 70)\n    tc("TOKEN CHECK FAILED: " + reason)\n    tc("The pooling positions are not settled, so no extraction")\n    tc("has run and no later number in this run can be trusted.")\n    tc("The full histogram above is the evidence. Show this whole")\n    tc("output to Muse.")\n    tc("!" * 70)\n    save_token_check()\n    sys.exit(11)\n\ndef audio_candidates():\n    # The candidate audio ids, from the model\'s own\n    # tokenizer and config, never a guessed number. Same\n    # evidence round 1 gathered; what changed is how the\n    # winner is picked (see token_check).\n    tok = processor.tokenizer\n    vocab = tok.get_vocab()\n    audio_named = {t: i for t, i in vocab.items() if "audio" in t}\n    tc("tokenizer tokens whose text mentions audio:")\n    for t in sorted(audio_named):\n        tc(f"  {t!r} -> id {audio_named[t]}")\n    candidates = {}\n    pad_id = tok.convert_tokens_to_ids("<|audio_pad|>")\n    if pad_id is not None and int(pad_id) >= 0:\n        candidates[int(pad_id)] = "<|audio_pad|> (converted by name)"\n    for t, i in audio_named.items():\n        candidates[int(i)] = f"tokenizer token {t!r}"\n    for holder_name, holder in (\n            ("model.config", model.config),\n            ("thinker_config",\n             getattr(model.config, "thinker_config", None))):\n        if holder is None:\n            continue\n        for attr in ("audio_token_id", "audio_token_index"):\n            val = getattr(holder, attr, None)\n            if val is not None:\n                tc(f"  {holder_name}.{attr} = {val}")\n                candidates.setdefault(\n                    int(val), f"config {holder_name}.{attr}")\n    return candidates\n\ndef histogram_for(label, path, prompt, candidates):\n    # Feed one master clip and print its full token\n    # histogram plus the processor\'s audio feature lengths.\n    # Returns (inputs, chosen_id, chosen_count, positions).\n    tok = processor.tokenizer\n    try:\n        inputs, audio_np = feed_inputs(path, prompt)\n    except FeedError as e:\n        token_check_fail(\n            f"the {label} clip\'s feed failed inside the token "\n            f"check: {e}")\n    row = [int(x) for x in inputs.input_ids[0].tolist()]\n    tc("-" * 70)\n    tc(f"TOKEN HISTOGRAM: {label} clip")\n    tc(f"  file: {path}")\n    tc(f"  decoded waveform: {len(audio_np)} samples at 16 kHz "\n       f"({len(audio_np) / 16000:.2f} s)")\n    tc(f"  sequence length: {len(row)} positions")\n    counts = Counter(row)\n    tc(f"  distinct token ids: {len(counts)}")\n    tc("  full histogram, most common first (every id with "\n       "count > 0):")\n    special_ids = set(int(x) for x in tok.all_special_ids)\n    for tid, cnt in counts.most_common():\n        name = tok.convert_ids_to_tokens([tid])[0]\n        decoded = tok.decode([tid])\n        flag = " [special token]" if tid in special_ids else ""\n        tc(f"    id {tid}: count {cnt}, token name {name!r}, "\n           f"decodes to {decoded!r}{flag}")\n    feats = getattr(inputs, "input_features", None)\n    if feats is not None:\n        tc(f"  processor input_features shape: "\n           f"{tuple(feats.shape)} (batch, mel bins, audio frames)")\n    else:\n        tc("  processor input_features: not present on the inputs")\n    fam = getattr(inputs, "feature_attention_mask", None)\n    if fam is not None:\n        tc(f"  processor feature_attention_mask shape: "\n           f"{tuple(fam.shape)}, true frames per row: "\n           f"{[int(x) for x in fam.sum(dim=-1).tolist()]}")\n    else:\n        tc("  processor feature_attention_mask: not present")\n    tc("  candidate audio ids, counted in this row:")\n    cand_counts = {}\n    for cid, why in sorted(candidates.items()):\n        cnt = row.count(cid)\n        cand_counts[cid] = cnt\n        tc(f"    candidate id {cid} ({why}): count {cnt}, "\n           f"decodes to {tok.decode([cid])!r}")\n    chosen = max(cand_counts, key=lambda c: cand_counts[c])\n    positions = [i for i, x in enumerate(row) if x == chosen]\n    tc(f"  DERIVED audio token id for the {label} clip: {chosen} "\n       f"({candidates[chosen]}), decodes to "\n       f"{tok.decode([chosen])!r}")\n    tc(f"  audio positions: {len(positions)} of {len(row)}; "\n       f"first {positions[:5]}, last {positions[-5:]}")\n    return inputs, chosen, cand_counts[chosen], positions\n\ndef token_check():\n    global AUDIO_TOKEN_ID, N_STATES, HIDDEN_SIZE\n    tc("=" * 70)\n    tc("TOKEN CHECK: settle the audio positions from the "\n       "histogram, before any extraction")\n    tc(f"plausibility floor: the derived audio-position count "\n       f"must be at least {MIN_AUDIO_POSITIONS}")\n    candidates = audio_candidates()\n    tone_path = os.path.join(CLIPS, "codec_tone_ref_raw.wav")\n    speech_path = os.path.join(CLIPS, "codec_speech_raw.wav")\n    tone_inputs, tone_id, tone_count, _ = histogram_for(\n        "tone", tone_path, PROMPTS["tone"], candidates)\n    _, speech_id, speech_count, _ = histogram_for(\n        "speech", speech_path, PROMPTS["speech"], candidates)\n    if tone_id != speech_id:\n        token_check_fail(\n            f"the two clips derive different audio token ids "\n            f"(tone {tone_id}, speech {speech_id}); the pooling "\n            f"positions cannot be settled on a split answer")\n    if tone_count < MIN_AUDIO_POSITIONS \\\n            or speech_count < MIN_AUDIO_POSITIONS:\n        token_check_fail(\n            f"the derived audio-position count is implausible "\n            f"(tone {tone_count}, speech {speech_count}, floor "\n            f"{MIN_AUDIO_POSITIONS}); the histogram does not "\n            f"show an id occurring at audio-frame scale")\n    AUDIO_TOKEN_ID = tone_id\n    tc("-" * 70)\n    tc(f"AUDIO TOKEN ID SETTLED: {AUDIO_TOKEN_ID}, from the "\n       f"histogram (tone count {tone_count}, speech count "\n       f"{speech_count})")\n    out = thinker_forward(tone_inputs)\n    hs = getattr(out, "hidden_states", None)\n    if hs is None or len(hs) == 0:\n        token_check_fail(\n            "the thinker returned no hidden states on the tone "\n            "clip; there would be nothing to pool")\n    N_STATES = len(hs)\n    HIDDEN_SIZE = int(hs[0].shape[2])\n    tc(f"hidden states returned: {N_STATES} tensors (layer 0 is "\n       f"the embedding output, then one per thinker layer)")\n    tc(f"thinker layers: {N_STATES - 1}")\n    tc(f"hidden size: {HIDDEN_SIZE}")\n    tc(f"tone sequence length: {int(hs[0].shape[1])}")\n    del out, tone_inputs\n    free_memory()\n    tc("TOKEN CHECK PASSED: the audio positions are derived "\n       "from the histogram, counted, and the same for both "\n       "clips. Every later number in this run pools over "\n       "these positions.")\n    save_token_check()\n\ntoken_check()\n\ndef audio_positions(input_ids_row):\n    row = [int(x) for x in input_ids_row]\n    return [i for i, x in enumerate(row) if x == AUDIO_TOKEN_ID]\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, and the token check")\n    print("all passed. No clips were generated and no extraction ran.")\n    print("Run Step 4 for the battery.")\n    sys.exit(0)\n\n\nimport csv  # the property head keeps its csv import in the\n# probe tail this runner cuts off; the first version of this\n# battery died on exactly that (NameError after the preflight\n# print, Nov 4, 2026). Imports the appended parts need live\n# here, not in the borrowed head.\n\n# --- Scorers and questions -------------------------------------------------------\n# The original behavioral questions (index check Part B); the\n# wording battery (Nov 4, 2026) showed their margins rank the\n# properties at AUC 1.0000 (pitch) and 0.9988 (onset), so they\n# are the readouts. The reversed pitch wording degraded its\n# own margin (0.6331) and is not used.\nYES_TOKEN_IDS = []\nfor _word in ("Yes", " Yes"):\n    _enc = processor.tokenizer.encode(_word,\n                                      add_special_tokens=False)\n    if _enc:\n        YES_TOKEN_IDS.append(int(_enc[0]))\nYES_TOKEN_IDS = sorted(set(YES_TOKEN_IDS))\nNO_TOKEN_IDS = []\nfor _word in ("No", " No"):\n    _enc = processor.tokenizer.encode(_word,\n                                      add_special_tokens=False)\n    if _enc:\n        NO_TOKEN_IDS.append(int(_enc[0]))\nNO_TOKEN_IDS = sorted(set(NO_TOKEN_IDS))\nPITCH_Q = ("Is the tone in this clip high-pitched? "\n           "Answer yes or no.")\nONSET_Q = ("Does the tone start in the first second of this "\n           "clip? Answer yes or no.")\nP3 = ("What sounds can you hear in this clip? Describe only "\n      "what is audible, without guessing from the pictures. "\n      "If the audio is silent, say \'silent\'.")\nprint("scorer tokens: yes", YES_TOKEN_IDS, "no", NO_TOKEN_IDS)\n\ndef share_and_word(logits):\n    probs = torch.softmax(logits.float(), dim=-1)\n    py = sum(float(probs[i].item()) for i in YES_TOKEN_IDS)\n    pn = sum(float(probs[i].item()) for i in NO_TOKEN_IDS)\n    share = py / (py + pn) if (py + pn) > 0 else 0.0\n    first = processor.tokenizer.decode(\n        [int(torch.argmax(logits).item())])\n    return share, first\n\ndef margins_for(path):\n    """Both questions\' margins and words, unpatched."""\n    out = {}\n    for key, q in (("pitch", PITCH_Q), ("onset", ONSET_Q)):\n        inputs, _ = feed_inputs(path, q, expect_silence=False)\n        o = thinker_forward(inputs)\n        share, first = share_and_word(o.logits[0, -1])\n        out[key] = (share, first)\n        del o, inputs\n        free_memory()\n    return out\n\ndef audio_block(inputs):\n    ids = inputs["input_ids"][0].tolist()\n    pos = [i for i, t in enumerate(ids) if t == AUDIO_TOKEN_ID]\n    if not pos:\n        print("PATCH HARD STOP: no audio positions in a feed.")\n        sys.exit(12)\n    if pos != list(range(pos[0], pos[-1] + 1)):\n        print("PATCH HARD STOP: audio positions not contiguous.")\n        sys.exit(12)\n    return pos\n\ndef post_positions(inputs):\n    pos = audio_block(inputs)\n    last = int(inputs["input_ids"].shape[1]) - 1\n    return list(range(pos[-1] + 1, last))\n\ndef patched_share(recip_path, donor_path, layer, mode,\n                  question):\n    """One patched forward: the donor\'s state at decoder\n    layer `layer` (tuple index layer+1) replaces the\n    recipient\'s rows at the mode\'s positions, during the\n    prefill pass only. Returns the question\'s yes share and\n    first word from the patched logits."""\n    r_in, _ = feed_inputs(recip_path, question,\n                          expect_silence=False)\n    d_in, _ = feed_inputs(donor_path, question,\n                          expect_silence=False)\n    if int(r_in["input_ids"].shape[1]) != int(\n            d_in["input_ids"].shape[1]):\n        print("PATCH HARD STOP: donor and recipient rows "\n              "differ in length.")\n        sys.exit(12)\n    d_out = thinker_forward(d_in)\n    d_state = d_out.hidden_states[layer + 1][0].detach().clone()\n    pos = (audio_block(r_in) if mode == "audio"\n           else post_positions(r_in))\n    pos_t = torch.tensor(pos, device=d_state.device)\n    prefill_len = int(r_in["input_ids"].shape[1])\n    layer_mod = THINKER.model.layers[layer]\n\n    def hook(module, args, output):\n        hs = output[0] if isinstance(output, tuple) else output\n        if hs.shape[1] != prefill_len:\n            return output\n        new = hs.clone()\n        new[0, pos_t] = d_state[pos_t].to(new.dtype)\n        if isinstance(output, tuple):\n            return (new,) + output[1:]\n        return new\n\n    handle = layer_mod.register_forward_hook(hook)\n    try:\n        o = thinker_forward(r_in)\n    finally:\n        handle.remove()\n    share, first = share_and_word(o.logits[0, -1])\n    del o, r_in, d_in, d_out, d_state\n    free_memory()\n    return share, first\n\ndef patched_openform(recip_path, donor_path, layer):\n    """The open question (P3) generated with the audio-position\n    patch held through the prefill pass."""\n    r_in, _ = feed_inputs(recip_path, P3, expect_silence=False)\n    d_in, _ = feed_inputs(donor_path, P3, expect_silence=False)\n    d_out = thinker_forward(d_in)\n    d_state = d_out.hidden_states[layer + 1][0].detach().clone()\n    pos = audio_block(r_in)\n    pos_t = torch.tensor(pos, device=d_state.device)\n    prefill_len = int(r_in["input_ids"].shape[1])\n    layer_mod = THINKER.model.layers[layer]\n\n    def hook(module, args, output):\n        hs = output[0] if isinstance(output, tuple) else output\n        if hs.shape[1] != prefill_len:\n            return output\n        new = hs.clone()\n        new[0, pos_t] = d_state[pos_t].to(new.dtype)\n        if isinstance(output, tuple):\n            return (new,) + output[1:]\n        return new\n\n    handle = layer_mod.register_forward_hook(hook)\n    try:\n        # The surgery\'s proven recipe, copied exactly (the\n        # handover rerun\'s generate_from_inputs): the FULL\n        # model\'s generate, no_grad, dict output, batch_decode.\n        # This function\'s first version called\n        # THINKER.generate with a bare tensor decode and\n        # failed on all 18 cells (Nov 4, 2026); the thinker\n        # submodule is not the generation entry point the\n        # surgery used.\n        with torch.no_grad():\n            out = model.generate(\n                **r_in, thinker_max_new_tokens=40,\n                do_sample=False, return_dict_in_generate=True,\n                output_scores=True)\n    finally:\n        handle.remove()\n    new_tokens = out.sequences[:, prefill_len:]\n    text = processor.batch_decode(\n        new_tokens, skip_special_tokens=True,\n        clean_up_tokenization_spaces=False)[0].strip()\n    del out, new_tokens, r_in, d_in, d_out, d_state\n    free_memory()\n    return text\n\n\n# --- The pair plan -----------------------------------------------------------------\n# Deterministic from the step-5 manifest. Cells are\n# (pitch_band, onset_band); pairing is by sorted clip name,\n# donors offset by one inside the matched cell, so a donor is\n# never its recipient\'s own clip.\nprint("=" * 70)\nprint("THE PROPERTY PATCH")\nprint("=" * 70)\n# Preflight (added after the first battery attempt died at\n# this exact spot with no trace in the log, Nov 4, 2026: the\n# Step 2 copies had not landed in that session). The battery\n# heals itself from the mounted Drive if it can, and says so\n# in the log either way; if it cannot, it stops here with the\n# reason printed, exit 13.\nimport shutil as _shutil\n_need_manifest = not os.path.exists(\n    "/content/property_manifest.csv")\n_need_clips = (not os.path.isdir("/content/property_clips")\n               or len([n for n in os.listdir(\n                   "/content/property_clips")\n                   if n.endswith(".wav")]) < 400)\nif _need_manifest or _need_clips:\n    print("preflight: property files missing in this "\n          "session; copying from the mounted Drive...")\n    try:\n        if _need_manifest:\n            _shutil.copy(\n                "/content/drive/MyDrive/tracing-qwen3b/"\n                "property_manifest.csv",\n                "/content/property_manifest.csv")\n        if _need_clips:\n            os.makedirs("/content/property_clips",\n                        exist_ok=True)\n            _src = ("/content/drive/MyDrive/tracing-qwen3b/"\n                    "property_clips")\n            for _n in os.listdir(_src):\n                if _n.endswith(".wav") and not os.path.exists(\n                        os.path.join("/content/property_clips",\n                                     _n)):\n                    _shutil.copy(os.path.join(_src, _n),\n                                 "/content/property_clips")\n        print("preflight: healed from Drive.")\n    except Exception as _e:\n        print("!" * 70)\n        print("PREFLIGHT STOP: the property files are not in "\n              "place and the Drive copy failed:")\n        print(repr(_e))\n        print("Step 2 of the notebook did not complete in "\n              "this session. Show this log to Muse.")\n        print("!" * 70)\n        sys.exit(13)\n_nwavs = len([n for n in os.listdir("/content/property_clips")\n              if n.endswith(".wav")])\nprint(f"preflight: manifest present, {_nwavs} property "\n      f"clips in place")\nwith open("/content/property_manifest.csv", newline="") as fh:\n    manifest = [r for r in csv.DictReader(fh)\n                if r["group"] == "tone"]\ncells_map = {}\nfor r in manifest:\n    cells_map.setdefault((r["pitch_band"], r["onset_band"]),\n                         []).append(r["clip_name"])\nfor k in cells_map:\n    cells_map[k] = sorted(cells_map[k])\nprint("cells:", {k: len(v) for k, v in cells_map.items()})\n\ndef clip_path(name):\n    return os.path.join("/content/property_clips",\n                        name + ".wav")\n\nLAYERS = [0, 14, 16, 18, 19, 20, 22, 24, 28, 34]\nPOST_LAYERS = [0, 19, 28]\nOPEN_LAYERS = [14, 19, 24]\nPAIRS_PER_CELL = 3\n\nplan = []   # (arm, direction, mode, recip, donor)\ndef add_pairs(arm, direction, mode, recip_cell, donor_cell,\n              n=PAIRS_PER_CELL):\n    recips = cells_map[recip_cell][:n]\n    donors = cells_map[donor_cell]\n    for i, recip in enumerate(recips):\n        donor = donors[(i + 1) % len(donors)]\n        if donor == recip:\n            donor = donors[(i + 2) % len(donors)]\n        plan.append((arm, direction, mode, recip, donor))\n\nfor onset in ("early", "late"):\n    add_pairs("pitch", "low_to_high", "audio",\n              ("low", onset), ("high", onset))\n    add_pairs("pitch", "high_to_low", "audio",\n              ("high", onset), ("low", onset))\n    add_pairs("pitch", "sham_low", "audio",\n              ("low", onset), ("low", onset))\n    add_pairs("pitch", "low_to_high", "post",\n              ("low", onset), ("high", onset))\nfor pitch in ("low", "high"):\n    add_pairs("onset", "early_to_late", "audio",\n              (pitch, "early"), (pitch, "late"))\n    add_pairs("onset", "late_to_early", "audio",\n              (pitch, "late"), (pitch, "early"))\n    add_pairs("onset", "sham_early", "audio",\n              (pitch, "early"), (pitch, "early"))\n    add_pairs("onset", "early_to_late", "post",\n              (pitch, "early"), (pitch, "late"))\nprint(f"pair plan: {len(plan)} pairs")\n\n# Baselines for every clip the plan touches.\nnames = sorted({p[3] for p in plan} | {p[4] for p in plan})\nbaselines = {}\ntruth = {r["clip_name"]: (r["pitch_band"], r["onset_band"])\n         for r in manifest}\nfor name in names:\n    baselines[name] = margins_for(clip_path(name))\nprint(f"baselines read for {len(baselines)} clips")\n\nPP_FIELDS = ["arm", "direction", "mode", "layer", "recipient",\n             "donor", "question", "base_share",\n             "donor_base_share", "share_after", "first_word",\n             "word_matches_donor"]\npp_rows = []\n\ndef run_cell(arm, direction, mode, layer, recip, donor):\n    for qkey, question in (("pitch", PITCH_Q),\n                           ("onset", ONSET_Q)):\n        share, first = patched_share(\n            clip_path(recip), clip_path(donor), layer, mode,\n            question)\n        donor_val = (truth[donor][0] == "high" if qkey == "pitch"\n                     else truth[donor][1] == "early")\n        word_yes = first.strip().lower().startswith("yes")\n        pp_rows.append({\n            "arm": arm, "direction": direction, "mode": mode,\n            "layer": layer, "recipient": recip, "donor": donor,\n            "question": qkey,\n            "base_share": f"{baselines[recip][qkey][0]:.4f}",\n            "donor_base_share":\n                f"{baselines[donor][qkey][0]:.4f}",\n            "share_after": f"{share:.4f}",\n            "first_word": first,\n            "word_matches_donor": word_yes == donor_val})\n\ntotal_cells = 0\nfor arm, direction, mode, recip, donor in plan:\n    layers = POST_LAYERS if mode == "post" else LAYERS\n    for layer in layers:\n        run_cell(arm, direction, mode, layer, recip, donor)\n        total_cells += 1\n    print(f"  done {arm}/{direction}/{mode}: {recip} <- {donor}")\n\ndef write_pp(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=PP_FIELDS)\n        w.writeheader()\n        w.writerows(pp_rows)\nsave_both("proppatch_results.csv", write_pp)\nprint(f"patch cells run: {total_cells} (two questions each)")\n\n# Open form at the partial layers, pitch arm, low to high.\nOF_FIELDS = ["recipient", "donor", "layer", "answer"]\nof_rows = []\nfor arm, direction, mode, recip, donor in plan:\n    if not (arm == "pitch" and direction == "low_to_high"\n            and mode == "audio"):\n        continue\n    for layer in OPEN_LAYERS:\n        try:\n            text = patched_openform(clip_path(recip),\n                                    clip_path(donor), layer)\n        except Exception as _e:\n            # Never let one generation kill the battery again\n            # (Nov 4, 2026): the traceback goes to the LOG (file\n            # must be sys.stdout; the default stderr is not\n            # teed, which is why the first failure left no\n            # trace) and the row itself carries the reason.\n            import traceback as _tb\n            _tb.print_exc(file=sys.stdout)\n            text = f"ERROR {_e.__class__.__name__}: {_e}"[:200]\n        of_rows.append({"recipient": recip, "donor": donor,\n                        "layer": layer, "answer": text})\n        print(f"  open form {recip} <- {donor} @ {layer}: "\n              f"{text!r}")\n\ndef write_of(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=OF_FIELDS)\n        w.writeheader()\n        w.writerows(of_rows)\nsave_both("proppatch_openform.csv", write_of)\n\nprint("PROPPATCH READOUT (mechanical):")\nfor arm in ("pitch", "onset"):\n    for direction in sorted({p[1] for p in plan if p[0] == arm}):\n        for mode in ("audio", "post"):\n            rs = [r for r in pp_rows\n                  if r["arm"] == arm\n                  and r["direction"] == direction\n                  and r["mode"] == mode\n                  and r["question"] == arm]\n            if not rs:\n                continue\n            line = []\n            for layer in sorted({int(r["layer"]) for r in rs}):\n                sel = [r for r in rs\n                       if int(r["layer"]) == layer]\n                moved = sum(\n                    abs(float(r["share_after"])\n                        - float(r["base_share"]))\n                    for r in sel) / len(sel)\n                toward = sum(\n                    (float(r["share_after"])\n                     - float(r["base_share"]))\n                    * (1 if float(r["donor_base_share"])\n                       > float(r["base_share"]) else -1)\n                    for r in sel) / len(sel)\n                line.append(f"L{layer}: move {moved:.3f} "\n                            f"toward {toward:+.3f}")\n            print(f"  {arm}/{direction}/{mode} (own question): "\n                  + "; ".join(line))\nprint("battery done: the property patch cells ran (audio, "\n      "sham, and post-position controls), with open-form cells "\n      "at the partial layers.")\nprint("Outputs: proppatch_results.csv, "\n      "proppatch_openform.csv, proppatch_run_log.txt, plus "\n      "the head\'s token check files.")\n'
with open("/content/qwen_proppatch_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_proppatch_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, token check --------------------------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, or the token check failed, and the run
# stops in this cell. The first check run downloads the model
# files once; that takes a while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_proppatch_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, or the token check.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, token check")
print("passed with the audio positions derived and counted. Go on to Step 4.")

## Step 4 · Run the battery

Battery mode reads baselines for every clip in the pair plan,
then runs the patch cells (two questions each), then the
open-form cells. A full run is long (roughly half an hour or
more). Results land in the Drive folder `tracing-qwen3b`:
`proppatch_results.csv`, `proppatch_openform.csv`,
`proppatch_run_log.txt`.


In [ ]:
# --- Battery mode: the tracing round 2 battery -----------------------------------------
# Same runner, same session Python, streamed live. The runner
# saves its outputs itself, to the Drive tracing folder and
# to /content. A nonzero exit code means the token check
# failed, a feed hard-stopped the run, clip generation failed
# its measurements, or the extraction count mismatched (the
# banners above say which).
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_proppatch_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: the token check failed, a feed proof hard-stopped")
    print("the run, clip generation failed, or the counts mismatched. Show")
    print("it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: token check passed, features extracted and counted,")
print("helpers trained with 5-shuffle controls, attention read, CSVs saved.")

## Closing · What to send back

From the Drive folder `tracing-qwen3b`, the run's own words:

1. the PROPPATCH READOUT block;
2. the battery exit code;
3. whether `proppatch_results.csv` and
   `proppatch_openform.csv` exist in the folder.
